In [14]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

# 1. Load the exact master file configuration
df = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# 2. Inverted rank target initialization
df['Consensus_Target'] = -df[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)
classic_features = ['Career_PPG', 'Championships', 'All_NBA_Teams', 'All_Defensive_Teams', 'Seasons_Played']

# 3. Handle data holes cleanly
df_clean = df.dropna(subset=['Consensus_Target']).copy()
df_clean['Career_PPG'] = df_clean['Career_PPG'].fillna(df_clean['Career_PPG'].mean())
df_clean['All_Defensive_Teams'] = df_clean['All_Defensive_Teams'].fillna(0)
df_clean['Championships'] = df_clean['Championships'].fillna(0)
df_clean['All_NBA_PS'] = df_clean['All_NBA_PS'].fillna(0)
df_clean['Seasons_Played'] = df_clean['Seasons_Played'].fillna(df_clean['Seasons_Played'].median())

# 4. Fit the Random Forest Ensemble
X = df_clean[classic_features]
y = df_clean['Consensus_Target']

rf_model = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
rf_model.fit(X, y)

# 5. Extract and print structural weights
rf_importances = rf_model.feature_importances_

print("🌳 RANDOM FOREST WEIGHT EXTRACTION COMPLETE")
print("-" * 45)
for feat, imp in sorted(zip(classic_features, rf_importances), key=lambda x: x[1], reverse=True):
    print(f"Metrics Element -> {feat:<20} | Weight: {imp*100:6.2f}%")

🌳 RANDOM FOREST WEIGHT EXTRACTION COMPLETE
---------------------------------------------
Metrics Element -> All_NBA_Teams        | Weight:  82.21%
Metrics Element -> Championships        | Weight:   7.96%
Metrics Element -> Career_PPG           | Weight:   5.07%
Metrics Element -> Seasons_Played       | Weight:   3.91%
Metrics Element -> All_Defensive_Teams  | Weight:   0.85%


In [15]:
#Creating our model using the Random Forest Model

import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Load the data
df_master = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# 2. Take a copy of the full pool from your loaded master dataframe
df_all_eras = df_master.copy()

# 3. Re-verify the classic parameters
classic_features = ['Career_PPG', 'Championships', 'All_NBA_Teams', 'All_Defensive_Teams', 'Seasons_Played']

# Clean structural missing values across eras safely
df_all_eras['Career_PPG'] = df_all_eras['Career_PPG'].fillna(df_all_eras['Career_PPG'].mean())
df_all_eras['All_Defensive_Teams'] = df_all_eras['All_Defensive_Teams'].fillna(0)

# 4. Z-Score Scale across all 114 players simultaneously
scaler_global = StandardScaler()
scaled_global_cols = [f'{col}_Global_Scaled' for col in classic_features]
df_all_eras[scaled_global_cols] = scaler_global.fit_transform(df_all_eras[classic_features])

# 5. Compute Global Classic Greatness Index
df_all_eras['Global_Classic_Index'] = (
    df_all_eras['All_NBA_Teams_Global_Scaled'] * 0.82 +
    df_all_eras['Championships_Global_Scaled'] * 0.08 +
    df_all_eras['Career_PPG_Global_Scaled'] * 0.05 +
    df_all_eras['Seasons_Played_Global_Scaled'] * 0.05
)

# 6. Rank and Sort
df_all_eras['Global_Classic_Rank'] = df_all_eras['Global_Classic_Index'].rank(ascending=False, method='min').astype(int)
df_all_eras_sorted = df_all_eras.sort_values(by='Global_Classic_Rank')

df_all_eras_sorted.to_csv('Random_Forest_Model.csv', index=False)
print("Done")

Done


In [16]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Load files
rf_file = 'Random_Forest_Model.csv'
df_rf = pd.read_csv(rf_file)

df_master = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# 2. Get ground truth consensus target
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# 3. Align data (safely isolating just the rank values)
df_eval = pd.merge(df_rf[['Player_Name', 'Global_Classic_Rank']], df_master[['Player_Name', 'Consensus_Rank']], on='Player_Name')
df_eval['Rank_Delta'] = df_eval['Global_Classic_Rank'] - df_eval['Consensus_Rank']

# 4. Calculate stats
mae = mean_absolute_error(df_eval['Consensus_Rank'], df_eval['Global_Classic_Rank'])
rmse = np.sqrt(mean_squared_error(df_eval['Consensus_Rank'], df_eval['Global_Classic_Rank']))
r2 = r2_score(df_eval['Consensus_Rank'], df_eval['Global_Classic_Rank'])

print("🌲 =================== RANDOM FOREST DIAGNOSTICS =================== 🌲")
print(f"📈 Model R² Score (Variance Explained): {r2:.4f}")
print(f"🎯 Mean Absolute Error (Average Deviation): {mae:.2f} positions")
print(f"📉 Root Mean Squared Error (Outlier Penalty): {rmse:.2f} positions")
print("-" * 68)

# Show top 3 closest and top 3 furthest
print("\n🎯 TOP 3 CLOSEST PREDICTIONS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nsmallest(3).index][['Player_Name', 'Consensus_Rank', 'Global_Classic_Rank', 'Rank_Delta']].to_string(index=False))

print("\n⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:")
print(df_eval.loc[df_eval['Rank_Delta'].abs().nlargest(3).index][['Player_Name', 'Consensus_Rank', 'Global_Classic_Rank', 'Rank_Delta']].to_string(index=False))
print("=====================================================================")

🌲 =================== RANDOM FOREST DIAGNOSTICS =================== 🌲
📈 Model R² Score (Variance Explained): 0.7087
🎯 Mean Absolute Error (Average Deviation): 12.54 positions
📉 Root Mean Squared Error (Outlier Penalty): 16.59 positions
--------------------------------------------------------------------

🎯 TOP 3 CLOSEST PREDICTIONS:
     Player_Name  Consensus_Rank  Global_Classic_Rank  Rank_Delta
   Pete Maravich       86.333333                   86   -0.333333
   Nate Thurmond       97.333333                   97   -0.333333
Chauncey Billups       83.666667                   83   -0.666667

⚠️ TOP 3 HIGHEST VARIANCE OUTLIERS:
 Player_Name  Consensus_Rank  Global_Classic_Rank  Rank_Delta
George Mikan       44.000000                  103   59.000000
Kevin McHale       49.333333                  108   58.666667
  Wes Unseld       70.000000                  113   43.000000


In [17]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Load and prepare master dataset
df_master = pd.read_csv(r'..\..\Data Collection\nba_top_100_master.csv')

# Calculate real-world tracking baseline
df_master['Consensus_Rank'] = df_master[['Athletic_Raw_Rank', 'Yahoo_Raw_Rank', 'BR_Raw_Rank']].mean(axis=1)

# Clean missing parameters safely
classic_features = ['Career_PPG', 'Championships', 'All_NBA_Teams', 'All_Defensive_Teams', 'Seasons_Played']
df_rf_clean = df_master.copy()
df_rf_clean['Career_PPG'] = df_rf_clean['Career_PPG'].fillna(df_rf_clean['Career_PPG'].mean())
df_rf_clean['All_Defensive_Teams'] = df_rf_clean['All_Defensive_Teams'].fillna(0)
df_rf_clean['Championships'] = df_rf_clean['Championships'].fillna(0)
df_rf_clean['All_NBA_Teams'] = df_rf_clean['All_NBA_Teams'].fillna(0)
df_rf_clean['Seasons_Played'] = df_rf_clean['Seasons_Played'].fillna(df_rf_clean['Seasons_Played'].median())

# 2. Apply Z-Score Standardization
scaler_global = StandardScaler()
scaled_cols = [f'{col}_Global_Scaled' for col in classic_features]
df_rf_clean[scaled_cols] = scaler_global.fit_transform(df_rf_clean[classic_features])

# 3. Compute Random Forest Feature Weights Index (With Season Scaling Fix)
df_rf_clean['Global_Classic_Index'] = (
    df_rf_clean['All_NBA_Teams_Global_Scaled'] * 0.82 +
    df_rf_clean['Championships_Global_Scaled'] * 0.08 +
    df_rf_clean['Career_PPG_Global_Scaled'] * 0.05 +
    df_rf_clean['Seasons_Played_Global_Scaled'] * 0.04
)
df_rf_clean['Global_Classic_Rank'] = df_rf_clean['Global_Classic_Index'].rank(ascending=False, method='min').astype(int)

# 4. Filter out only the Top 20 for display
rf_top20 = df_rf_clean.sort_values(by='Global_Classic_Rank').head(20)[
    ['Global_Classic_Rank', 'Player_Name', 'Global_Classic_Index', 'Consensus_Rank']
]

# 5. Print Output Dashboard
print("\n🌲 ====================== RANDOM FOREST MODEL: TOP 20 ====================== 🌲")
print(f"{'RF Rank':<9} | {'Player Name':<22} | {'Model Index':<11} | {'Actual Consensus Rank':<21}")
print("-" * 75)
for _, row in rf_top20.iterrows():
    print(f"{int(row['Global_Classic_Rank']):<9} | {row['Player_Name']:<22} | {row['Global_Classic_Index']:<11.4f} | {row['Consensus_Rank']:<21.2f}")


🌲 ====================== RANDOM FOREST MODEL: TOP 20 ====================== 🌲
RF Rank   | Player Name            | Model Index | Actual Consensus Rank
---------------------------------------------------------------------------
1         | LeBron James           | 3.1016      | 2.00                 
2         | Kareem Abdul-Jabbar    | 2.1472      | 3.00                 
3         | Kobe Bryant            | 2.1106      | 9.33                 
4         | Tim Duncan             | 2.0361      | 6.33                 
5         | Shaquille O'Neal       | 1.8489      | 7.00                 
6         | Karl Malone            | 1.7808      | 18.00                
7         | Bob Cousy              | 1.4243      | 37.00                
8         | Bill Russell           | 1.3864      | 4.67                 
9         | Michael Jordan         | 1.3627      | 1.00                 
10        | John Havlicek          | 1.3595      | 30.67                
11        | Hakeem Olajuwon        | 1.344